In [0]:
from pathlib import Path

import requests
from databricks.sdk import WorkspaceClient

DEFAULT_AS_OF = spark.sql(
    "SELECT date_format(from_utc_timestamp(current_timestamp(), 'America/Toronto'), 'yyyy-MM-dd') AS d"
).first()["d"]

dbutils.widgets.text("catalog", "serverless_nv91sa_catalog")
dbutils.widgets.text("team", "team1")
dbutils.widgets.text("app_url", "")
dbutils.widgets.text("as_of_date", DEFAULT_AS_OF)

catalog = dbutils.widgets.get("catalog")
team = dbutils.widgets.get("team")
app_url = dbutils.widgets.get("app_url").rstrip("/")

response = requests.get(f"{app_url}/?page=health", timeout=60) if app_url else None
health_ok = bool(response and response.ok and "HEALTHCHECK: SUCCESS" in response.text)

# Alerting must live outside the browser session: a SQL alert or a scheduled job named for the team.
workspace = WorkspaceClient()
durable_alerts = []
try:
    durable_alerts += [a.display_name for a in workspace.alerts.list() if (a.display_name or "").lower().startswith(team.lower())]
except Exception as exc:
    print(f"SQL alert probe note: {exc}")
try:
    durable_alerts += [
        j.settings.name for j in workspace.jobs.list(name=None)
        if j.settings and j.settings.name and team in j.settings.name and "alert" in j.settings.name.lower() and j.settings.schedule
    ]
except Exception as exc:
    print(f"Job probe note: {exc}")

main_source = (Path.cwd().resolve().parents[0] / "app" / "main.py").read_text(encoding="utf-8")
browser_loop_removed = "time.sleep(ALERT_INTERVAL_SECONDS)" not in main_source

ok = health_ok and bool(durable_alerts) and browser_loop_removed
status = "PASS" if ok else "FAIL"
metric_value = 1.0 if ok else 0.0

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.ops")
spark.sql(f"CREATE TABLE IF NOT EXISTS {catalog}.ops.milestones (team STRING, round INT, status STRING, metric_name STRING, metric_value DOUBLE, ts TIMESTAMP)")
spark.sql(
    f"INSERT INTO {catalog}.ops.milestones VALUES ('{team}', 5, '{status}', 'durable_alerting', {metric_value}, current_timestamp())"
)

print({"status": status, "health_ok": health_ok, "durable_alerts": durable_alerts, "browser_loop_removed": browser_loop_removed})
